# So sanh voi YOLO: SSD-MobileNet (torchvision)
**Muc tieu (yeu cau thay):** train mot model KHAC HO YOLO nhung nhe, de so sanh voi YOLOv8n.
- Model: **SSDlite320 - MobileNetV3-Large** (torchvision) - kien truc SSD + MobileNet, khac han YOLO.
- Dataset: dung lai bo **70:15:15** da chia (doc thang nhan YOLO .txt).
- Do: mAP@0.5, mAP@0.5:0.95, toc do (FPS) de lap bang so sanh.

## 1. [SETUP] Cai dat & kiem tra GPU

In [ ]:
# ============================================================
# [SETUP] Kiem tra GPU + cai thu vien do mAP
# ============================================================
!nvidia-smi
!pip install torchmetrics pycocotools -q
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. [DATA] Mount Drive & copy dataset 70:15:15 ve local

In [ ]:
# ============================================================
# [DATA] Mount Drive + copy bo 70:15:15 ve o local Colab
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515"
dst = "/content/vehicles_7015"
if not os.path.exists(dst):
    print("Dang copy dataset... (cho vai phut)")
    shutil.copytree(src, dst)
print("Dataset san sang:", dst)
for sp in ["train","valid","test"]:
    p = os.path.join(dst, sp, "images")
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"{sp:6s}: {n} anh")

## 3. [DATASET] Lop doc du lieu YOLO cho torchvision
Chuyen nhan YOLO (class cx cy w h, chuan hoa) -> box [xmin,ymin,xmax,ymax] tuyet doi.
Torchvision danh so lop tu 1 (0 = background), nen: bus=1, car=2, motorcycle=3, truck=4.

In [ ]:
# ============================================================
# [DATASET] VehicleDataset - doc anh + nhan YOLO
# ============================================================
import os, torch
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as F

# Ten lop (giu dung thu tu voi YOLO); +1 vi torchvision danh so tu 1
CLASSES = ['bus', 'car', 'motorcycle', 'truck']

class VehicleDataset(Dataset):
    def __init__(self, root, split, img_size=320):
        self.img_dir = os.path.join(root, split, "images")
        self.lbl_dir = os.path.join(root, split, "labels")
        self.img_size = img_size
        self.files = [f for f in os.listdir(self.img_dir)
                      if f.lower().endswith(('.jpg','.jpeg','.png'))]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        W, H = img.size
        img = img.resize((self.img_size, self.img_size))

        boxes, labels = [], []
        lbl_path = os.path.join(self.lbl_dir, os.path.splitext(fname)[0] + ".txt")
        if os.path.exists(lbl_path):
            for line in open(lbl_path):
                parts = line.split()
                if len(parts) < 5:
                    continue
                cid, cx, cy, w, h = map(float, parts[:5])
                # YOLO chuan hoa -> toa do tuyet doi tren anh da resize
                x1 = (cx - w/2) * self.img_size
                y1 = (cy - h/2) * self.img_size
                x2 = (cx + w/2) * self.img_size
                y2 = (cy + h/2) * self.img_size
                x1, y1 = max(0,x1), max(0,y1)
                x2, y2 = min(self.img_size,x2), min(self.img_size,y2)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1,y1,x2,y2])
                    labels.append(int(cid) + 1)   # +1: background=0

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        return F.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

print("Da dinh nghia VehicleDataset")

## 4. [MODEL] Nap SSDlite320-MobileNetV3

In [ ]:
# ============================================================
# [MODEL] SSDlite320 MobileNetV3-Large (pretrained COCO)
# ============================================================
import torch
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.models.detection._utils import retrieve_out_channels
from functools import partial
import torch.nn as nn

NUM_CLASSES = len(CLASSES) + 1   # 4 lop + background = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Nap model pretrained tren COCO
model = ssdlite320_mobilenet_v3_large(weights="DEFAULT")

# Thay classification head cho 5 lop (4 xe + background)
in_channels = retrieve_out_channels(model.backbone, (320, 320))
num_anchors = model.anchor_generator.num_anchors_per_location()
norm_layer  = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
model.head.classification_head = SSDLiteClassificationHead(
    in_channels, num_anchors, NUM_CLASSES, norm_layer)

model.to(device)
print("Da nap SSDlite320-MobileNetV3 |", NUM_CLASSES, "lop | device:", device)
n_params = sum(p.numel() for p in model.parameters())
print(f"So tham so: {n_params/1e6:.2f} trieu")

## 5. [TRAIN] Huan luyen SSD-MobileNet
Train ~50 epoch (SSDlite nhe, nhanh). Luu best.pt theo mAP tren valid.

In [ ]:
# ============================================================
# [TRAIN] Vong lap huan luyen SSD-MobileNet
# ============================================================
import torch, time
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision

torch.manual_seed(42)

IMG_SIZE = 320
BATCH    = 16
EPOCHS   = 50
ROOT     = "/content/vehicles_7015"
SAVE_DIR = "/content/runs_ssd"
os.makedirs(SAVE_DIR, exist_ok=True)

train_ds = VehicleDataset(ROOT, "train", IMG_SIZE)
valid_ds = VehicleDataset(ROOT, "valid", IMG_SIZE)
train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, collate_fn=collate_fn)
valid_ld = DataLoader(valid_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate_fn)

optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_map = 0.0
history = []   # luu (epoch, train_loss, mAP50, mAP5095)

for epoch in range(1, EPOCHS+1):
    # ---- TRAIN ----
    model.train()
    running = 0.0
    for imgs, targets in train_ld:
        imgs = [im.to(device) for im in imgs]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]
        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item()
    scheduler.step()
    train_loss = running / len(train_ld)

    # ---- VALIDATE (mAP) ----
    model.eval()
    metric = MeanAveragePrecision(iou_thresholds=[0.5] + [round(0.5+0.05*i,2) for i in range(10)])
    with torch.no_grad():
        for imgs, targets in valid_ld:
            imgs = [im.to(device) for im in imgs]
            preds = model(imgs)
            preds = [{k:v.cpu() for k,v in p.items()} for p in preds]
            metric.update(preds, targets)
    res = metric.compute()
    mAP50   = float(res["map_50"])
    mAP5095 = float(res["map"])
    history.append((epoch, train_loss, mAP50, mAP5095))
    print(f"Epoch {epoch:3d}/{EPOCHS} | loss={train_loss:.3f} | mAP50={mAP50:.4f} | mAP50-95={mAP5095:.4f}")

    # ---- SAVE BEST ----
    if mAP50 > best_map:
        best_map = mAP50
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best.pt"))

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "last.pt"))
print(f"\nXong! Best mAP50 (valid) = {best_map:.4f}")

## 6. [EVAL] Danh gia tren tap TEST + do toc do (FPS)

In [ ]:
# ============================================================
# [EVAL] Danh gia best.pt tren tap TEST + do FPS
# ============================================================
import torch, time
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Nap best.pt
model.load_state_dict(torch.load("/content/runs_ssd/best.pt"))
model.eval()

test_ds = VehicleDataset(ROOT, "test", IMG_SIZE)
test_ld = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, collate_fn=collate_fn)

metric = MeanAveragePrecision(class_metrics=True,
    iou_thresholds=[0.5] + [round(0.5+0.05*i,2) for i in range(10)])

t0, n_imgs = time.time(), 0
with torch.no_grad():
    for imgs, targets in test_ld:
        imgs = [im.to(device) for im in imgs]
        preds = model(imgs)
        preds = [{k:v.cpu() for k,v in p.items()} for p in preds]
        metric.update(preds, targets)
        n_imgs += len(imgs)
elapsed = time.time() - t0
res = metric.compute()

print("===== SSD-MobileNet TREN TAP TEST =====")
print("mAP@0.5      =", round(float(res["map_50"]), 4))
print("mAP@0.5:0.95 =", round(float(res["map"]), 4))
print(f"Toc do       = {n_imgs/elapsed:.1f} FPS ({elapsed*1000/n_imgs:.1f} ms/anh)")

# mAP tung lop (neu co)
if "map_per_class" in res and res["map_per_class"].numel() > 1:
    print("\n===== mAP@0.5:0.95 TUNG LOP =====")
    for i, cname in enumerate(CLASSES):
        try:
            print(f"  {cname:12s}: {float(res['map_per_class'][i]):.4f}")
        except Exception:
            pass

## 7. [PLOT + SAVE] Ve bieu do & luu ket qua ve Drive

In [ ]:
# ============================================================
# [PLOT + SAVE] Bieu do loss/mAP + luu ve Drive
# ============================================================
import matplotlib.pyplot as plt, shutil, os

ep   = [h[0] for h in history]
loss = [h[1] for h in history]
m50  = [h[2] for h in history]
m5095= [h[3] for h in history]

fig, ax = plt.subplots(1, 2, figsize=(14,5))
ax[0].plot(ep, loss, color="#d62728", lw=2); ax[0].set_title("SSD-MobileNet: Train Loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].grid(True, alpha=0.3)
ax[1].plot(ep, m50,   label="mAP@0.5",      color="#2ca02c", lw=2)
ax[1].plot(ep, m5095, label="mAP@0.5:0.95", color="#ff7f0e", lw=2)
ax[1].set_title("SSD-MobileNet: mAP theo Epoch"); ax[1].set_xlabel("Epoch")
ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/runs_ssd/bieudo_ssd.png", dpi=150, bbox_inches="tight")
plt.show()

# Luu toan bo ket qua ve Drive
dst = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_SSD_MobileNet"
shutil.copytree("/content/runs_ssd", dst, dirs_exist_ok=True)
print("Da luu ket qua SSD ve Drive:", dst)

## 8. [COMPARE] Bang so sanh YOLOv8n vs SSD-MobileNet

In [ ]:
# ============================================================
# [COMPARE] Bang so sanh (dien so SSD tu cell [EVAL] o tren)
# ============================================================
import pandas as pd

# YOLOv8n: so lieu da co tu tuan truoc (tren tap test)
# SSD: LAY tu ket qua cell [EVAL] roi dien vao day
compare = pd.DataFrame([
    {"Model":"YOLOv8n",        "Kien truc":"YOLO 1-stage",   "mAP@0.5":0.686, "mAP@0.5:0.95":0.480, "FPS":"~450", "Tham so (M)":3.0},
    {"Model":"SSD-MobileNetV3","Kien truc":"SSD + MobileNet", "mAP@0.5":None,  "mAP@0.5:0.95":None,  "FPS":"...", "Tham so (M)":3.4},
])
print(compare.to_string(index=False))
print("\n=> Dien mAP/FPS cua SSD tu cell [EVAL] vao dong tuong ung.")